In [ ]:
import numpy as np
from subprocess import PIPE, run
import matplotlib.pyplot as plt
import os
import textwrap
from waxx.control import ethernet_relay

class ExptBuilder():
    def __init__(self):
        self.__code_path__ = os.environ.get('code')
        self.__temp_exp_path__ = os.path.join(self.__code_path__, "k-exp", "kexp", "experiments", "ml_expt.py")

    def run_expt(self):
        expt_path = self.__temp_exp_path__
        run_expt_command = r"%kpy% & ar " + expt_path
        result = run(run_expt_command, stdout=PIPE, stderr=PIPE, universal_newlines=True, shell=True)
        print(result.returncode, result.stdout, result.stderr)
        os.remove(self.__temp_exp_path__)
        return result.returncode
    
    def write_experiment_to_file(self, program):
        with open(self.__temp_exp_path__, 'w') as file:
            file.write(program)
    
    def fringe_scan_expt(self, img_amp, freq_raman):
        script = textwrap.dedent(f"""
        from artiq.experiment import *
        from artiq.experiment import delay
        from kexp import Base, img_types, cameras
        import numpy as np
        from kexp.calibrations.tweezer import tweezer_vpd1_to_vpd2
        from kexp.calibrations.imaging import high_field_imaging_detuning
        from artiq.coredevice.sampler import Sampler
        from artiq.language import now_mu
        from kexp.util.artiq.async_print import aprint

        class hf_monitored_rabi(EnvExperiment, Base):

            def prepare(self):
                Base.__init__(self,setup_camera=True,
                            camera_select=cameras.andor,
                            save_data=True,
                            imaging_type=img_types.DISPERSIVE)

                self.p.v_pd_hf_tweezer_squeeze_power = .984
                                 
                self.p.frequency_raman_transition = {freq_raman}

                self.p.t_continuous_rabi = 450.e-6
                
                self.p.amp_imaging = {img_amp}
                
                self.p.dimension_slm_mask = 20.e-6

                self.p.phase_slm_mask = 0.4 * np.pi

                self.p.t_tweezer_hold = 20.e-3

                self.p.t_tof = 20.e-6
                
                self.p.t_mot_load = 1.0
                
                self.p.N_repeats = 10

                self.scope = self.scope_data.add_siglent_scope("192.168.1.108", label='PD', arm=False)

                self.finish_prepare(shuffle=True)

            @kernel
            def scan_kernel(self):
                
                self.set_imaging_detuning(frequency_detuned = self.p.frequency_detuned_hf_midpoint)
                self.slm.write_phase_mask_kernel(phase=self.p.phase_slm_mask,dimension=self.p.dimension_slm_mask)
                self.imaging.set_power(self.p.amp_imaging)

                self.prepare_hf_tweezers(squeeze=True)

                self.raman.init(fraction_power = self.p.fraction_power_raman,
                                frequency_transition = self.p.frequency_raman_transition)

                self.ttl.raman_shutter.on()
                delay(10.e-3)
                self.ttl.line_trigger.wait_for_line_trigger()
                delay(4.7e-3)
                
                self.ttl.pd_scope_trig3.pulse(1.e-6)
                self.imaging.on()
                delay(3.e-6)

                self.raman.pulse(t=self.p.t_continuous_rabi)
                
                self.imaging.off()

                self.ttl.raman_shutter.off()
                
                self.set_imaging_detuning(frequency_detuned = self.p.frequency_detuned_hf_f1m1)
                self.imaging.set_power(.2,reset_pid=True)

                delay(self.p.t_tweezer_hold)
                self.tweezer.off()

                delay(self.p.t_tof)

                self.abs_image()

                self.core.wait_until_mu(now_mu())
                self.scope.read_sweep(0)
                self.core.break_realtime()
                delay(30.e-3)

            @kernel
            def run(self):
                self.init_kernel()
                self.load_2D_mot(self.p.t_2D_mot_load_delay)
                self.scan()
                self.mot_observe()

            def analyze(self):
                import os
                expt_filepath = os.path.abspath(__file__)
                # aprint(self.scope._data)
                self.end(expt_filepath)
                """)
        return script

In [6]:
eBuilder = ExptBuilder()

In [7]:
img_amp = np.linspace(.2,2.5,100)

freq_raman = np.array([1.19498693e+08, 1.19503061e+08, 1.19507429e+08, 1.19511796e+08,
       1.19516164e+08, 1.19520532e+08, 1.19524899e+08, 1.19529267e+08,
       1.19533635e+08, 1.19538002e+08, 1.19542370e+08, 1.19546738e+08,
       1.19551105e+08, 1.19555473e+08, 1.19559841e+08, 1.19564208e+08,
       1.19568576e+08, 1.19572944e+08, 1.19577312e+08, 1.19581679e+08,
       1.19586047e+08, 1.19590415e+08, 1.19594782e+08, 1.19599150e+08,
       1.19603518e+08, 1.19607885e+08, 1.19612253e+08, 1.19616621e+08,
       1.19620988e+08, 1.19625356e+08, 1.19629724e+08, 1.19634091e+08,
       1.19638459e+08, 1.19642827e+08, 1.19647194e+08, 1.19651562e+08,
       1.19655930e+08, 1.19660297e+08, 1.19664665e+08, 1.19669033e+08,
       1.19673400e+08, 1.19677768e+08, 1.19682136e+08, 1.19686503e+08,
       1.19690871e+08, 1.19695239e+08, 1.19699606e+08, 1.19703974e+08,
       1.19708342e+08, 1.19712709e+08, 1.19717077e+08, 1.19721445e+08,
       1.19725813e+08, 1.19730180e+08, 1.19734548e+08, 1.19738916e+08,
       1.19743283e+08, 1.19747651e+08, 1.19752019e+08, 1.19756386e+08,
       1.19760754e+08, 1.19765122e+08, 1.19769489e+08, 1.19773857e+08,
       1.19778225e+08, 1.19782592e+08, 1.19786960e+08, 1.19791328e+08,
       1.19795695e+08, 1.19800063e+08, 1.19804431e+08, 1.19808798e+08,
       1.19813166e+08, 1.19817534e+08, 1.19821901e+08, 1.19826269e+08,
       1.19830637e+08, 1.19835004e+08, 1.19839372e+08, 1.19843740e+08,
       1.19848107e+08, 1.19852475e+08, 1.19856843e+08, 1.19861211e+08,
       1.19865578e+08, 1.19869946e+08, 1.19874314e+08, 1.19878681e+08,
       1.19883049e+08, 1.19887417e+08, 1.19891784e+08, 1.19896152e+08,
       1.19900520e+08, 1.19904887e+08, 1.19909255e+08, 1.19913623e+08,
       1.19917990e+08, 1.19922358e+08, 1.19926726e+08, 1.19931093e+08])


for f in range(len(img_amp)):
    print(img_amp[f],freq_raman[f])
    eBuilder.write_experiment_to_file(eBuilder.fringe_scan_expt(img_amp=img_amp[f],freq_raman = freq_raman[f]))
    eBuilder.run_expt()

0.2 119498693.0
0  10 values of dummy. 10 total shots. 30 total images expected.
Run ID: 68670
Acknowledged camera ready signal.
Camera is ready.

Sent: {'mask': 'spot', 'center': [994, 820], 'phase': 0.4, 'dimension': 20, 'initialize': False, 'spacing': 10, 'angle': 45}
-> mask: spot, dimension = 20 um, phase = 0.4 pi, x-center = 994, y-center = 820

 Run ID: 68670

Sent: {'mask': 'spot', 'center': [994, 820], 'phase': 0.4, 'dimension': 20, 'initialize': False, 'spacing': 10, 'angle': 45}
-> mask: spot, dimension = 20 um, phase = 0.4 pi, x-center = 994, y-center = 820

Wavemeter exposure time 581 ms is too long for reliable lock detection.
Wavemeter exposure time 581 ms is too long for reliable lock detection.
Error getting frequency from wavemeter
laser ry_405 unlocked:
target = 741.091200, meas = 0.000000, diff = -741091200.0
shot 1/10 done

Sent: {'mask': 'spot', 'center': [994, 820], 'phase': 0.4, 'dimension': 20, 'initialize': False, 'spacing': 10, 'angle': 45}
-> mask: spot, dim